# 3. SQLite storage and SQL queries

We store the processed BFS municipality records, cleaned individual tariffs and municipality comparison values in SQLite. We run the SQL queries from Python.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

working_directory = Path.cwd()

project_directory = (
    working_directory.parent
    if working_directory.name == "notebooks"
    else working_directory
)

processed_directory = project_directory / "data" / "processed"

print("Data directory", processed_directory)
print("Directory exists", processed_directory.is_dir())

Data directory data/processed
Directory exists True


In [2]:
# Load the complete BFS municipality reference data.
df_municipalities = pd.read_csv(
    processed_directory / "bfs_municipalities_2026.csv",
    dtype={"municipality_number": "Int64"}
)

# Load the cleaned individual tariff values.
df_tariffs = pd.read_csv(
    processed_directory / "cleaned_tariffs.csv",
    dtype={"municipality_number_2026": "Int64"}
)

# Load one comparison tariff per municipality.
df_municipality_values = pd.read_csv(
    processed_directory / "municipality_values.csv",
    dtype={"municipality_number_2026": "Int64"}
)

print("Municipality reference records", len(df_municipalities))
print("Individual tariff values", len(df_tariffs))
print("Municipality comparison values", len(df_municipality_values))

Municipality reference records 2110
Individual tariff values 2255
Municipality comparison values 2108


## 3.1 Create the database tables

The database file is `data/electricity_tariffs.sqlite`. The `municipalities` table stores municipality names, cantons and municipality types. The `tariffs` table stores cleaned tariff values. The `municipality_values` table stores one comparison value per municipality with a valid tariff. The municipality number links all three tables.

In [3]:
# Select municipality reference fields.
df_municipalities_sql = df_municipalities[
    [
        "municipality_number",
        "municipality_name",
        "canton_id",
        "canton",
        "municipality_type_code",
        "municipality_type"
    ]
].copy()

# Use a consistent municipality key for individual tariffs.
df_tariffs_sql = (
    df_tariffs.rename(
        columns={"municipality_number_2026": "municipality_number"}
    )[
        ["municipality_number", "operator_uri", "total_tariff"]
    ]
    .copy()
)

# Select municipality comparison values and their keys.
df_municipality_values_sql = (
    df_municipality_values.rename(
        columns={"municipality_number_2026": "municipality_number"}
    )[
        ["municipality_number", "municipality_tariff"]
    ]
    .copy()
)

# Validate keys before writing to the database.
assert df_municipalities_sql["municipality_number"].notna().all()
assert df_municipalities_sql["municipality_number"].is_unique

assert df_municipality_values_sql["municipality_number"].notna().all()
assert df_municipality_values_sql["municipality_number"].is_unique

assert df_tariffs_sql["municipality_number"].isin(
    df_municipalities_sql["municipality_number"]
).all()

assert df_municipality_values_sql["municipality_number"].isin(
    df_municipalities_sql["municipality_number"]
).all()

print("Prepared database tables.")

Prepared database tables.


In [4]:
from contextlib import closing

db_path = project_directory / "data" / "electricity_tariffs.sqlite"

with closing(sqlite3.connect(db_path)) as connection:

    df_municipalities_sql.to_sql(
        "municipalities",
        connection,
        if_exists="replace",
        index=False
    )

    df_tariffs_sql.to_sql(
        "tariffs",
        connection,
        if_exists="replace",
        index=False
    )

    df_municipality_values_sql.to_sql(
        "municipality_values",
        connection,
        if_exists="replace",
        index=False
    )

print("Saved database", db_path)

Saved database data/electricity_tariffs.sqlite


In [5]:
sql = """
SELECT 'municipalities' AS table_name, COUNT(*) AS count
FROM municipalities

UNION ALL

SELECT 'tariffs' AS table_name, COUNT(*) AS count
FROM tariffs

UNION ALL

SELECT 'municipality_values' AS table_name, COUNT(*) AS count
FROM municipality_values;
"""

with closing(sqlite3.connect(db_path)) as connection:
    database_counts = pd.read_sql_query(sql, connection)

display(database_counts)

,table_name,count
0,municipalities,2110
1,tariffs,2255
2,municipality_values,2108


## 3.2 Join and summarize tables with SQL

We join `municipality_values` to `municipalities` on the municipality number. For each municipality type, we calculate the number of municipalities and the mean, minimum and maximum comparison tariff. Each municipality counts once. The SQL summary is unadjusted. The planned regression will account for canton.

In [6]:
sql_join = """
SELECT
    g.municipality_number,
    g.municipality_name,
    g.canton_id,
    g.canton,
    g.municipality_type_code,
    g.municipality_type,
    w.municipality_tariff

FROM municipality_values AS w

INNER JOIN municipalities AS g
    ON w.municipality_number = g.municipality_number

ORDER BY g.municipality_number;
"""

with closing(sqlite3.connect(db_path)) as connection:
    df_analysis_sql = pd.read_sql_query(
        sql_join,
        connection
    )

print("Municipalities after the SQL join", len(df_analysis_sql))

assert len(df_analysis_sql) == len(df_municipality_values_sql)
assert df_analysis_sql["municipality_number"].is_unique

display(df_analysis_sql.head())

Municipalities after the SQL join 2108


,municipality_number,municipality_name,canton_id,canton,municipality_type_code,municipality_type,municipality_tariff
0,1,Aeugst am Albis,1,Zürich,3,Rural,24.136
1,2,Affoltern am Albis,1,Zürich,1,Urban,24.136
2,3,Bonstetten,1,Zürich,2,Intermediate,24.136
3,4,Hausen am Albis,1,Zürich,2,Intermediate,24.136
4,5,Hedingen,1,Zürich,1,Urban,24.136


In [7]:
sql_by_type = """
SELECT
    g.municipality_type_code,
    g.municipality_type,
    COUNT(*) AS municipality_count,
    ROUND(AVG(w.municipality_tariff), 3) AS mean_rp_kwh,
    ROUND(MIN(w.municipality_tariff), 3) AS min_rp_kwh,
    ROUND(MAX(w.municipality_tariff), 3) AS max_rp_kwh

FROM municipality_values AS w

INNER JOIN municipalities AS g
    ON w.municipality_number = g.municipality_number

GROUP BY
    g.municipality_type_code,
    g.municipality_type

ORDER BY g.municipality_type_code;
"""

with closing(sqlite3.connect(db_path)) as connection:
    df_by_type_sql = pd.read_sql_query(
        sql_by_type,
        connection
    )

assert df_by_type_sql["municipality_count"].sum() == len(
    df_analysis_sql
)

display(df_by_type_sql)

,municipality_type_code,municipality_type,municipality_count,mean_rp_kwh,min_rp_kwh,max_rp_kwh
0,1,Urban,508,28.005,21.883,36.910
1,2,Intermediate,579,28.182,16.227,41.365
2,3,Rural,1021,28.361,9.643,43.613


## 3.3 Find municipalities without a comparison tariff

We use a left join to keep every BFS municipality and select missing matches with `IS NULL`. We expect Augst and Nürensdorf, as identified during preparation. We also check that the municipalities with and without comparison values together account for the full BFS register.

In [8]:
sql_without_tariffs = """
SELECT
    g.municipality_number,
    g.municipality_name,
    g.canton,
    g.municipality_type

FROM municipalities AS g

LEFT JOIN municipality_values AS w
    ON g.municipality_number = w.municipality_number

WHERE w.municipality_number IS NULL

ORDER BY g.municipality_number;
"""

with closing(sqlite3.connect(db_path)) as connection:
    df_without_tariffs_sql = pd.read_sql_query(
        sql_without_tariffs,
        connection
    )

display(df_without_tariffs_sql)

assert len(df_analysis_sql) + len(df_without_tariffs_sql) == len(
    df_municipalities_sql
)

,municipality_number,municipality_name,canton,municipality_type
0,64,Nürensdorf,Zürich,Intermediate
1,2822,Augst,Basel-Landschaft,Urban


## 3.4 Save SQL queries and results

We save the four SQL queries in `sql/` and export the table counts, municipality-type summary and municipalities without a comparison tariff as CSV files in `results/sql/`.

In [9]:
sql_directory = project_directory / "sql"
results_directory = project_directory / "results" / "sql"

sql_directory.mkdir(parents=True, exist_ok=True)
results_directory.mkdir(parents=True, exist_ok=True)

# Save SQL queries.
queries = {
    "01_table_counts.sql": sql,
    "02_municipality_values_join.sql": sql_join,
    "03_tariffs_by_municipality_type.sql": sql_by_type,
    "04_municipalities_without_tariffs.sql": sql_without_tariffs
}

for filename, query in queries.items():
    (sql_directory / filename).write_text(
        query,
        encoding="utf-8"
    )

# Save results.
database_counts.to_csv(
    results_directory / "table_counts.csv",
    index=False
)

df_by_type_sql.to_csv(
    results_directory / "tariffs_by_municipality_type.csv",
    index=False
)

df_without_tariffs_sql.to_csv(
    results_directory / "municipalities_without_tariffs.csv",
    index=False
)

print("Saved SQL queries and results.")

Saved SQL queries and results.
